# Counts Calculator Tutorial: Pattern Frequency Analysis with BeowulfFullLines.txt

## Introduction

The `Counts` calculator counts how many times specific patterns appear in each window of text. It's the foundation of rolling windows analysis - giving you raw frequency data that shows how patterns change throughout your document.

## Setup

In [1]:
import spacy
from lexos.rolling_windows import Windows
from lexos.rolling_windows.calculators import Counts
import pandas as pd

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Load and prepare BeowulfFullLines.txt
with open("BeowulfFullLines.txt", "r", encoding="utf-8") as f:
    beowulf_text = f.read()

# Apply scrubbing (matching web app)
import string
text = beowulf_text.lower()
text = ''.join(char for char in text if not char.isdigit())
punct = string.punctuation
text = ''.join(char for char in text if char not in punct or char in 'ðþ')

print(f"Loaded and scrubbed Beowulf text: {len(text)} characters")

ImportError: cannot import name 'Counts' from 'lexos.rolling_windows.calculators' (C:\Users\aaron\uv_lexos\src\lexos\rolling_windows\calculators\__init__.py)

## Basic Pattern Counting

In [ ]:
# Create windows
windows = Windows()
analysis_windows = windows(
    input=text,
    n=1000,
    window_type="characters",
    output="strings"
)

# Count Old English characters (matching web app test)
counter = Counts(
    patterns=["ð", "þ"],
    windows=analysis_windows,
    mode="exact",
    case_sensitive=False
)

# Get results
results = counter.to_df()
print("First 5 windows - character counts:")
print(results.head())
print(f"\nTotal ð occurrences: {results['ð'].sum()}")
print(f"Total þ occurrences: {results['þ'].sum()}")

## Understanding Parameters

### patterns
The patterns to search for:

In [ ]:
# Single pattern
single_windows = windows(input=text[:5000], n=500, window_type="characters")
single_counter = Counts(patterns=["ð"], windows=single_windows)
print("Single pattern:", single_counter.to_df().columns.tolist())

# Multiple patterns
multi_windows = windows(input=text[:5000], n=500, window_type="characters")
multi_counter = Counts(patterns=["ð", "þ", "æ"], windows=multi_windows)
print("Multiple patterns:", multi_counter.to_df().columns.tolist())

### mode
How to search for patterns:

In [ ]:
# Process text for different modes
doc = nlp(text[:10000])

# Exact mode (default)
exact_windows = windows(input=doc, n=100, window_type="tokens", output="strings")
exact_counter = Counts(
    patterns=["wæs"],
    windows=exact_windows,
    mode="exact"
)
print(f"Exact 'wæs' matches: {exact_counter.to_df()['wæs'].sum()}")

# Regex mode
regex_windows = windows(input=doc, n=100, window_type="tokens", output="strings")
regex_counter = Counts(
    patterns=["^wear.*"],  # Words starting with "wear"
    windows=regex_windows,
    mode="regex"
)
print(f"Regex '^wear.*' matches: {regex_counter.to_df()['^wear.*'].sum()}")

### case_sensitive
Control case matching:

In [ ]:
# Case sensitive
sensitive_windows = windows(input=text[:1000], n=200, window_type="characters")
sensitive_counter = Counts(
    patterns=["þ", "Þ"],  # Lowercase and uppercase
    windows=sensitive_windows,
    case_sensitive=True
)
print("Case sensitive results:")
print(f"  þ (lowercase): {sensitive_counter.to_df()['þ'].sum()}")
print(f"  Þ (uppercase): {sensitive_counter.to_df()['Þ'].sum()}")

# Case insensitive
insensitive_windows = windows(input=text[:1000], n=200, window_type="characters")
insensitive_counter = Counts(
    patterns=["þ"],
    windows=insensitive_windows,
    case_sensitive=False
)
print(f"\nCase insensitive þ: {insensitive_counter.to_df()['þ'].sum()}")

## Search Modes in Detail

### Exact Mode
Finds exact matches only:

In [ ]:
test_text = "Beowulf saw the beast. The beastly creature was beaten."
test_doc = nlp(test_text.lower())

exact_windows = windows(input=test_doc, n=10, window_type="tokens", output="strings")
exact_counter = Counts(
    patterns=["beast"],
    windows=exact_windows,
    mode="exact"
)
print("Exact mode - only 'beast':")
print(exact_counter.to_df())

### Regex Mode
Uses regular expressions:

In [ ]:
regex_windows = windows(input=test_doc, n=10, window_type="tokens", output="strings")
regex_counter = Counts(
    patterns=["beast.*"],  # Matches beast, beastly
    windows=regex_windows,
    mode="regex"
)
print("\nRegex mode - 'beast.*':")
print(regex_counter.to_df())

### SpaCy Rule Mode
Linguistic pattern matching:

In [ ]:
# Need token output for spaCy rules
spacy_windows = windows(input=test_doc, n=10, window_type="tokens", output="tokens")
spacy_counter = Counts(
    patterns=[[{"POS": "NOUN"}]],  # Find all nouns
    windows=spacy_windows,
    mode="spacy_rule",
    model="en_core_web_sm"
)
print("\nSpaCy rule mode - all nouns:")
print(spacy_counter.to_df())

## Replicating Web App Tests

### Test 1: Character Patterns (ð,þ)

In [ ]:
# Matching web app exactly
char_windows = windows(
    input=text,
    n=1000,
    window_type="characters",
    output="strings"
)

char_counter = Counts(
    patterns=["ð", "þ"],
    windows=char_windows,
    mode="exact"
)

results = char_counter.to_df()
print(f"Windows with both characters: {((results['ð'] > 0) & (results['þ'] > 0)).sum()}")
print(f"Average ð per window: {results['ð'].mean():.2f}")
print(f"Average þ per window: {results['þ'].mean():.2f}")

### Test 2: Word Search

In [ ]:
# Process for word searching
doc = nlp(text[:50000])

word_windows = windows(
    input=doc,
    n=1000,
    window_type="tokens",
    output="strings"
)

word_counter = Counts(
    patterns=["wæs"],
    windows=word_windows,
    mode="exact"
)

word_results = word_counter.to_df()
print(f"Windows containing 'wæs': {(word_results['wæs'] > 0).sum()}")
print(f"Total 'wæs' occurrences: {word_results['wæs'].sum()}")

### Test 3: Regex Patterns

In [ ]:
# Regex search matching web app
regex_windows = windows(
    input=doc,
    n=1000,
    window_type="tokens",
    output="strings"
)

regex_counter = Counts(
    patterns=["^wear.*"],
    windows=regex_windows,
    mode="regex"
)

regex_results = regex_counter.to_df()
print(f"Windows with words starting 'wear': {(regex_results['^wear.*'] > 0).sum()}")

## Working with Different Window Types

### Character Windows

In [ ]:
# Character-based analysis
char_analysis = windows(
    input=text[:2000],
    n=100,
    window_type="characters",
    output="strings"
)

punctuation_counter = Counts(
    patterns=[".", ",", "!", "?"],
    windows=char_analysis,
    mode="exact"
)

print("Punctuation frequency in character windows:")
print(punctuation_counter.to_df().sum())

### Token Windows

In [ ]:
# Token-based analysis
doc_sample = nlp(text[:5000])
token_analysis = windows(
    input=doc_sample,
    n=50,
    window_type="tokens",
    output="strings"
)

common_words = Counts(
    patterns=["and", "the", "of", "in"],
    windows=token_analysis,
    mode="exact"
)

print("\nCommon word frequency in token windows:")
print(common_words.to_df().sum())

## Analyzing Results

### Getting DataFrame Output

In [ ]:
# The to_df() method returns a pandas DataFrame
df = counter.to_df()

print("DataFrame structure:")
print(f"  Rows (windows): {len(df)}")
print(f"  Columns (patterns): {list(df.columns)}")
print(f"  Data type: {type(df)}")

# Access specific data
print(f"\nFirst window ð count: {df.iloc[0]['ð']}")
print(f"Last window þ count: {df.iloc[-1]['þ']}")

### Statistical Analysis

In [ ]:
# Analyze pattern distribution
stats = {
    'Pattern': df.columns,
    'Total': df.sum().values,
    'Mean': df.mean().values,
    'Max': df.max().values,
    'Windows_with_pattern': (df > 0).sum().values
}

stats_df = pd.DataFrame(stats)
print("\nPattern statistics:")
print(stats_df)

## Important Notes

### Generators Get Consumed

In [ ]:
# Wrong way - reusing windows
test_windows = windows(input=text[:1000], n=100, window_type="characters")
counter1 = Counts(patterns=["ð"], windows=test_windows)
# This will fail - windows already consumed!
# counter2 = Counts(patterns=["þ"], windows=test_windows)

# Right way - fresh windows for each use
windows1 = windows(input=text[:1000], n=100, window_type="characters")
counter1 = Counts(patterns=["ð"], windows=windows1)

windows2 = windows(input=text[:1000], n=100, window_type="characters")
counter2 = Counts(patterns=["þ"], windows=windows2)

print("Always create fresh windows for each calculator!")

### Matching Window and Pattern Types

In [ ]:
# SpaCy rules need token output
doc = nlp("Test text")

# Wrong - string output with spaCy rules
# string_windows = windows(input=doc, n=5, window_type="tokens", output="strings")
# bad_counter = Counts(patterns=[[{"POS": "NOUN"}]], windows=string_windows, mode="spacy_rule")

# Right - token output with spaCy rules
token_windows = windows(input=doc, n=5, window_type="tokens", output="tokens")
good_counter = Counts(patterns=[[{"POS": "NOUN"}]], windows=token_windows, mode="spacy_rule")

print("SpaCy rules require token output!")

## Summary

The Counts calculator:
- **Counts pattern occurrences** in each window
- **Supports multiple search modes**: exact, regex, spaCy rules
- **Returns pandas DataFrames** for easy analysis
- **Forms the foundation** for Averages and Ratios calculators

### Next Steps

Use count data with:

In [ ]:
# Calculate averages (normalized frequencies)
from lexos.rolling_windows.calculators import Averages
avg_calc = Averages(patterns=["ð", "þ"], windows=your_windows)

# Visualize results
from lexos.rolling_windows.plotters import SimplePlotter
plotter = SimplePlotter()
plotter.plot(df)